# Fig. 10 & 11: Latent-Space PCA, Unsupervised Clustering, and LASSO Interpretation

This notebook reproduces **Figure 10** (PCA of the CVAE latent means with unsupervised
Gaussian-mixture clustering that recovers the four physical regimes label-free) and
**Figure 11** (LASSO regularization paths relating PC2/PC3 to the physical state vector),
for both the Holton--Mass model and the emulator.

PCA is fit **independently for each model**. Clustering uses only latent coordinates;
the physical regime labels (AA, BB, AB, BA) are used afterward only to color and score
the clusters. Weights are the canonical paper checkpoint (`checkpoint_11`).

## Setup
Model, data paths, thresholds, sampling, and shared helper functions.

In [1]:
import os, sys
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, r2_score
from sklearn.linear_model import Lasso, LinearRegression
from scipy.optimize import linear_sum_assignment

plt.rcParams.update({
    "font.size": 15, "axes.titlesize": 17, "axes.labelsize": 15,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 13,
    "figure.titlesize": 19,
})

# ---- paths (update to your environment) ----
DATA_HM_PATH   = r"/home/danyul/ssw/stochastic_trajectories.npy"               # (N, 2, 75)
DATA_PRED_PATH = r"/home/danyul/ssw/predictions_best_checkpoint_and_cycle_Resnet_VAE_3mil.npy"  # (N, 75)
MODEL_PATH     = r"/home/danyul/ssw/work_torch/checkpoint_11.pth"              # canonical paper weights
OUT_DIR        = r"../graphs_for_paper"   # bundle output folder for paper PNGs
os.makedirs(OUT_DIR, exist_ok=True)

# import the bundle's model definition (model.py lives at the repo root)
for up in ("..", os.path.join("..", ".."), "."):
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), up)))
from model import ConditionalVAE

LATENT_DIM, OUTPUT_DIM, CONDITION_DIM = 32, 75, 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConditionalVAE(LATENT_DIM, OUTPUT_DIM, CONDITION_DIM).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print("Model loaded on", device)

# ---- constants ----
UPPER = 53.8 / 2.8935   # u_A threshold (normalized)
LOWER = 7.41            # u_B threshold (normalized)
N_USE, PER_CLASS, SEED = 2_500_000, 750, 2025
BATCH_SIZE = 8192
LABEL_ORDER  = ["A", "B", "A->B", "B->A"]
LABEL_COLORS = {"A": "blue", "B": "red", "A->B": "green", "B->A": "purple"}
# monospace labels matching the manuscript notation (AA/BB/AB/BA)
DISPLAY = {"A": r"$\mathtt{AA}$", "B": r"$\mathtt{BB}$",
           "A->B": r"$\mathtt{AB}$", "B->A": r"$\mathtt{BA}$"}
# IHF constants (paper formula: integrate e^{-z/H}[Re psi dz Im psi - Im psi dz Re psi] to 30 km)
H_KM, Z_TOP_KM, N_LEVELS = 7.0, 70.0, 25
DZ = Z_TOP_KM / N_LEVELS
RE, IM, US = slice(0, 25), slice(25, 50), slice(50, 75)


def detect_transitions_A_to_B(u, upper, lower):
    out, i = [], 1
    while i < len(u) - 1:
        if u[i - 1] > upper and u[i] <= upper:
            j = i + 1
            while j < len(u) and u[j] <= upper:
                if u[j] < lower:
                    out.append(i); break
                j += 1
            i = j
        else:
            i += 1
    return np.array(out, dtype=np.int64)


def detect_transitions_B_to_A(u, upper, lower):
    out, i = [], 1
    while i < len(u) - 1:
        if u[i - 1] < lower and u[i] >= lower:
            j = i + 1
            while j < len(u) and u[j] >= lower:
                if u[j] > upper:
                    out.append(i); break
                j += 1
            i = j
        else:
            i += 1
    return np.array(out, dtype=np.int64)


def build_balanced(zonal_wind, getter, n_use, per_class, lo_pad=0.0, hi_pad=0.0):
    """Balanced PER_CLASS sample of A / B / A->B / B->A using only the
    zonal-wind regime definitions; getter(indices) returns the 75-d states."""
    rng = np.random.default_rng(SEED)
    tA = detect_transitions_A_to_B(zonal_wind, UPPER, LOWER)
    tB = detect_transitions_B_to_A(zonal_wind, UPPER, LOWER)
    union = np.union1d(tA, tB)
    idx = np.arange(n_use)
    nonA = np.where((zonal_wind > UPPER + hi_pad) & (~np.isin(idx, union)))[0]
    nonB = np.where((zonal_wind < LOWER - lo_pad) & (~np.isin(idx, union)))[0]
    n = min(per_class, len(tA), len(tB), len(nonA), len(nonB))
    sub = lambda a: a if len(a) <= n else rng.choice(a, size=n, replace=False)
    tA, tB, nonA, nonB = sub(tA), sub(tB), sub(nonA), sub(nonB)
    X = np.vstack([getter(nonA), getter(nonB), getter(tA), getter(tB)]).astype(np.float32)
    labels = np.array(["A"] * len(nonA) + ["B"] * len(nonB)
                      + ["A->B"] * len(tA) + ["B->A"] * len(tB))
    return X, labels


def batched_encode(X):
    mus = []
    with torch.no_grad():
        for i in range(0, len(X), BATCH_SIZE):
            xb = torch.from_numpy(X[i:i + BATCH_SIZE]).to(device)
            mu, _ = model.encode(xb)
            mus.append(mu.cpu().numpy())
    return np.concatenate(mus, axis=0)


def hungarian_map(true_labels, cl):
    """Best one-to-one cluster->regime assignment; returns (map, accuracy)."""
    cl_ids = np.unique(cl)
    cm = np.zeros((len(cl_ids), len(LABEL_ORDER)), dtype=int)
    for i, c in enumerate(cl_ids):
        for j, lab in enumerate(LABEL_ORDER):
            cm[i, j] = np.sum((cl == c) & (true_labels == lab))
    r, cc = linear_sum_assignment(cm.max() - cm)
    mapping = {cl_ids[ri]: LABEL_ORDER[ci] for ri, ci in zip(r, cc)}
    return mapping, cm[r, cc].sum() / len(true_labels)


def compute_IHF(data):
    re, im = data[:, RE], data[:, IM]
    z = np.linspace(DZ, Z_TOP_KM, N_LEVELS)
    dRe = np.empty_like(re); dIm = np.empty_like(im)
    dRe[:, 1:-1] = (re[:, 2:] - re[:, :-2]) / (2 * DZ)
    dIm[:, 1:-1] = (im[:, 2:] - im[:, :-2]) / (2 * DZ)
    dRe[:, 0] = (re[:, 1] - re[:, 0]) / DZ; dRe[:, -1] = (re[:, -1] - re[:, -2]) / DZ
    dIm[:, 0] = (im[:, 1] - im[:, 0]) / DZ; dIm[:, -1] = (im[:, -1] - im[:, -2]) / DZ
    core = re * dIm - im * dRe
    w = np.exp(-z / H_KM)[None, :]
    kmax = max(1, min(N_LEVELS - 1, int(np.floor(30.0 / DZ)) - 1))
    return np.trapezoid(w[:, :kmax + 1] * core[:, :kmax + 1], dx=DZ, axis=1)


print("Setup complete.")


Model loaded on cuda
Setup complete.


## Figure 10: PCA + unsupervised GMM clustering
For each model we encode a balanced sample of the four regimes, fit PCA independently,
fit a 4-component Gaussian mixture model on the PC1--PC2 plane, and match clusters to
regimes with the Hungarian algorithm. The right column shows the GMM clusters
(black 1$\sigma$/2$\sigma$ covariance ellipses) colored by the regime they best match.

In [2]:
def orient(pcs, labels):
    """Sign-normalize: A at +PC1, A->B at +PC2 (match published layout)."""
    a, b, ab = labels == "A", labels == "B", labels == "A->B"
    if pcs[a, 0].mean() < pcs[b, 0].mean():
        pcs[:, 0] *= -1
    if pcs[ab, 1].mean() < 0:
        pcs[:, 1] *= -1
    return pcs


def draw_ellipse(ax, mean, cov):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    for nsig in (1, 2):
        w, h = 2 * nsig * np.sqrt(vals)
        ax.add_patch(Ellipse(mean, w, h, angle=angle, edgecolor="black",
                             facecolor="none", lw=2.0, ls="--", alpha=0.9, zorder=4))


# load data (memory-mapped) and encode each model ONCE; reused by later cells
hm  = np.load(DATA_HM_PATH, mmap_mode="r")
emu = np.load(DATA_PRED_PATH, mmap_mode="r")

DATA = {}
for tag, zonal, getter, pads in [
    ("Holton-Mass Model", np.asarray(hm[:N_USE, 0, 63]),
     lambda idx: np.asarray(hm[idx, 0]), dict()),
    ("Emulator", np.asarray(emu[:N_USE, 63]),
     lambda idx: np.asarray(emu[idx]), dict(lo_pad=5.0, hi_pad=5.0)),
]:
    X, y = build_balanced(zonal, getter, N_USE, PER_CLASS, **pads)
    mu = batched_encode(X)
    pca = PCA(n_components=3).fit(mu)             # PCA fit INDEPENDENTLY per model
    pcs = pca.transform(mu)
    evr = pca.explained_variance_ratio_
    pc = orient(pcs[:, :2].copy(), y)            # 2-D plane for the figure
    gm = GaussianMixture(n_components=4, covariance_type="full",
                         n_init=5, random_state=SEED).fit(pc)
    gm_cl = gm.predict(pc)
    gm_map, gm_acc = hungarian_map(y, gm_cl)
    proba = gm.predict_proba(pc).max(1)
    DATA[tag] = dict(X=X, y=y, mu=mu, evr=evr, pc=pc, gm=gm, gm_cl=gm_cl,
                     gm_map=gm_map, gm_acc=gm_acc, proba=proba)
    print(f"{tag:20s} EVR PC1-3 = {100*evr[0]:.2f}/{100*evr[1]:.2f}/{100*evr[2]:.2f}%  "
          f"| GMM(PC1-2) match = {100*gm_acc:.1f}%")

# paper figure: physical regimes (left) vs unsupervised GMM clusters (right), 2 rows
fig, ax = plt.subplots(2, 2, figsize=(15, 12))
for r, tag in enumerate(["Holton-Mass Model", "Emulator"]):
    d = DATA[tag]
    at = ax[r, 0]
    for reg in LABEL_ORDER:
        m = d["y"] == reg
        at.scatter(d["pc"][m, 0], d["pc"][m, 1], s=18, alpha=0.30, lw=0,
                   c=LABEL_COLORS[reg], label=DISPLAY[reg])
        at.scatter(d["pc"][m, 0].mean(), d["pc"][m, 1].mean(), marker="x",
                   s=260, c="black", linewidths=3.0, zorder=5)
    at.set_title(f"{tag}: physical regimes")
    at.set_xlabel("PC1"); at.set_ylabel("PC2"); at.grid(alpha=0.2)
    at.legend(markerscale=2.5, loc="best")
    ac = ax[r, 1]
    for cid in np.unique(d["gm_cl"]):
        reg = d["gm_map"].get(cid, "?")
        m = d["gm_cl"] == cid
        ac.scatter(d["pc"][m, 0], d["pc"][m, 1], s=18, alpha=0.30, lw=0,
                   c=LABEL_COLORS.get(reg, "grey"))
    for cid in range(d["gm"].n_components):
        draw_ellipse(ac, d["gm"].means_[cid], d["gm"].covariances_[cid])
    ac.set_title(f"{tag}: GMM clusters ({100*d['gm_acc']:.0f}% match)")
    ac.set_xlabel("PC1"); ac.set_ylabel("PC2"); ac.grid(alpha=0.2)
for k, a in enumerate(ax.ravel()):
    a.text(0.02, 0.98, f"({chr(97 + k)})", transform=a.transAxes,
           fontsize=20, fontweight="bold", va="top", ha="left")
fig.tight_layout(pad=1.5)
fig.savefig(os.path.join(OUT_DIR, "gmm_regimes_vs_clusters.png"),
            dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved Fig 10 -> gmm_regimes_vs_clusters.png")

# posterior assignment confidence (PC1-2 GMM) — backs the text's soft BB/BA numbers
for tag in ["Holton-Mass Model", "Emulator"]:
    d = DATA[tag]
    print(f"\n{tag}: GMM(PC1-2) max posterior probability by regime")
    for reg in LABEL_ORDER:
        m = d["y"] == reg
        print(f"   {DISPLAY[reg]:14s} mean={d['proba'][m].mean():.2f} "
              f"median={np.median(d['proba'][m]):.2f} frac>0.9={np.mean(d['proba'][m] > 0.9):.2f}")


Holton-Mass Model    EVR PC1-3 = 99.60/0.17/0.09%  | GMM(PC1-2) match = 89.1%


Emulator             EVR PC1-3 = 99.64/0.14/0.09%  | GMM(PC1-2) match = 87.1%


Saved Fig 10 -> gmm_regimes_vs_clusters.png

Holton-Mass Model: GMM(PC1-2) max posterior probability by regime
   $\mathtt{AA}$  mean=0.96 median=1.00 frac>0.9=0.87
   $\mathtt{BB}$  mean=0.89 median=0.90 frac>0.9=0.51
   $\mathtt{AB}$  mean=0.97 median=1.00 frac>0.9=0.90
   $\mathtt{BA}$  mean=0.92 median=1.00 frac>0.9=0.71

Emulator: GMM(PC1-2) max posterior probability by regime
   $\mathtt{AA}$  mean=0.99 median=1.00 frac>0.9=0.98
   $\mathtt{BB}$  mean=0.90 median=0.92 frac>0.9=0.69
   $\mathtt{AB}$  mean=1.00 median=1.00 frac>0.9=1.00
   $\mathtt{BA}$  mean=0.91 median=0.95 frac>0.9=0.71


## Unsupervised model selection and full-latent purity
The silhouette score (a label-free criterion) is maximized at $k=4$ on the standardized
first three PCs for both models, but selects $k=2$ when only PC1 is supplied. Cluster
accuracy improves to $\approx 96\%$ (emulator) / $\approx 89\%$ (Holton--Mass) when the
full 32-dimensional latent space is used, with AA/AB separated at $\approx 98$--$100\%$
purity and the weak-vortex classes BB/BA somewhat mixed.

In [3]:
print("\n" + "=" * 60 + "\nUnsupervised model selection & full-latent purity\n" + "=" * 60)
ks = list(range(2, 9))
for tag in ["Holton-Mass Model", "Emulator"]:
    d = DATA[tag]
    mu, y = d["mu"], d["y"]
    pcs3 = PCA(n_components=3).fit_transform(mu)
    pc3_std = StandardScaler().fit_transform(pcs3)
    mu_std = StandardScaler().fit_transform(mu)

    sil_std = [silhouette_score(pc3_std, KMeans(k, n_init=10, random_state=SEED).fit_predict(pc3_std)) for k in ks]
    sil_pc1 = [silhouette_score(pcs3[:, :1], KMeans(k, n_init=10, random_state=SEED).fit_predict(pcs3[:, :1])) for k in ks]

    # headline accuracy uses the standardized full 32-d latent
    gm32 = GaussianMixture(4, covariance_type="full", n_init=5, random_state=SEED).fit(mu_std)
    cl32 = gm32.predict(mu_std)
    map32, acc32 = hungarian_map(y, cl32)

    print(f"\n{tag}:")
    print(f"   silhouette best k  | standardized PC1-3 = {ks[int(np.argmax(sil_std))]}"
          f"   PC1-only = {ks[int(np.argmax(sil_pc1))]}")
    print(f"   full 32-d (std) GMM accuracy = {100*acc32:.1f}%")
    print("   per-cluster purity:")
    for cid in np.unique(cl32):
        comp = {lab: int(np.sum((cl32 == cid) & (y == lab))) for lab in LABEL_ORDER}
        tot = sum(comp.values()); dom = max(comp, key=comp.get)
        print(f"     cluster {cid} -> {DISPLAY[map32.get(cid)]}: purity={comp[dom]/tot:.2f}  {comp}")

# silhouette-vs-k figure (optional supporting plot)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for tag, mk in zip(["Holton-Mass Model", "Emulator"], ["o-", "s-"]):
    pcs3 = PCA(n_components=3).fit_transform(DATA[tag]["mu"])
    pc3_std = StandardScaler().fit_transform(pcs3)
    sil_std = [silhouette_score(pc3_std, KMeans(k, n_init=10, random_state=SEED).fit_predict(pc3_std)) for k in ks]
    sil_pc1 = [silhouette_score(pcs3[:, :1], KMeans(k, n_init=10, random_state=SEED).fit_predict(pcs3[:, :1])) for k in ks]
    ax[0].plot(ks, sil_std, mk, label=tag)
    ax[1].plot(ks, sil_pc1, mk, label=tag)
ax[0].axvline(4, color="grey", ls="--", alpha=0.6); ax[0].set_title("standardized PC1-3"); ax[0].set_xlabel("k"); ax[0].set_ylabel("silhouette"); ax[0].legend(fontsize=10)
ax[1].axvline(2, color="grey", ls="--", alpha=0.6); ax[1].set_title("PC1 only"); ax[1].set_xlabel("k"); ax[1].legend(fontsize=10)
fig.suptitle("Silhouette score vs number of clusters")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "silhouette_model_selection.png"), dpi=180, bbox_inches="tight")
plt.close(fig)
print("\nSaved silhouette_model_selection.png")



Unsupervised model selection & full-latent purity



Holton-Mass Model:
   silhouette best k  | standardized PC1-3 = 4   PC1-only = 2
   full 32-d (std) GMM accuracy = 89.3%
   per-cluster purity:
     cluster 0 -> $\mathtt{BB}$: purity=0.86  {'A': 0, 'B': 580, 'A->B': 0, 'B->A': 96}
     cluster 1 -> $\mathtt{AB}$: purity=0.94  {'A': 46, 'B': 0, 'A->B': 742, 'B->A': 0}
     cluster 2 -> $\mathtt{BA}$: purity=0.79  {'A': 0, 'B': 170, 'A->B': 0, 'B->A': 654}
     cluster 3 -> $\mathtt{AA}$: purity=0.99  {'A': 704, 'B': 0, 'A->B': 8, 'B->A': 0}



Emulator:
   silhouette best k  | standardized PC1-3 = 4   PC1-only = 2
   full 32-d (std) GMM accuracy = 95.7%
   per-cluster purity:
     cluster 0 -> $\mathtt{BA}$: purity=0.98  {'A': 0, 'B': 1, 'A->B': 13, 'B->A': 643}
     cluster 1 -> $\mathtt{AB}$: purity=0.99  {'A': 7, 'B': 0, 'A->B': 736, 'B->A': 0}
     cluster 2 -> $\mathtt{AA}$: purity=1.00  {'A': 743, 'B': 0, 'A->B': 0, 'B->A': 0}
     cluster 3 -> $\mathtt{BB}$: purity=0.87  {'A': 0, 'B': 749, 'A->B': 1, 'B->A': 107}



Saved silhouette_model_selection.png


## Figure 11: LASSO interpretation of PC2 / PC3
We regress the emulator's PC2 and PC3 onto the 75 state variables with LASSO. Both are
largely linear in the state (OLS $R^2 = 0.91$, $0.94$). PC3 admits a compact $\sim$10-term
description and is moderately correlated with IHF(30 km) ($r=0.66$, $0.42$ after removing the
linear $U$(30 km) dependence); PC2 spreads across many zonal-wind levels. PC1 is essentially
$U$(30 km) ($r=0.98$).

In [4]:
print("\n" + "=" * 60 + "\nLASSO interpretation of PC2 / PC3\n" + "=" * 60)
rng = np.random.default_rng(SEED)
mu_e = DATA["Emulator"]["mu"]
pca3 = PCA(n_components=3).fit(mu_e)            # 3-comp PCA on emulator latent means
samp = np.sort(rng.choice(1_000_000, 12000, replace=False))
Xs = np.asarray(emu[samp]).astype(np.float32)
mus = batched_encode(Xs)
pcs = pca3.transform(mus)
Xstd = StandardScaler().fit_transform(Xs)
ntr = 9000
Xtr, Xte = Xstd[:ntr], Xstd[ntr:]

# correlations reported in text
U30 = Xs[:, 63]
ihf = compute_IHF(Xs)
r_pc1_u = np.corrcoef(pcs[:, 0], U30)[0, 1]
r_pc3_ihf = np.corrcoef(pcs[:, 2], ihf)[0, 1]
def residual(a, b):
    b1 = np.column_stack([np.ones_like(b), b])
    return a - b1 @ np.linalg.lstsq(b1, a, rcond=None)[0]
r_pc3_ihf_partial = np.corrcoef(residual(pcs[:, 2], U30), residual(ihf, U30))[0, 1]
print(f"r(PC1, U30) = {r_pc1_u:.3f}")
print(f"r(PC3, IHF) = {r_pc3_ihf:.3f}   partial (control U30) = {r_pc3_ihf_partial:.3f}")

# full linear-model (OLS) R^2 of each PC on the 75-d state -> "largely linear"
for idx, lab in [(1, "PC2"), (2, "PC3")]:
    yy = pcs[:, idx]; yy = (yy - yy.mean()) / yy.std()
    ols_r2 = LinearRegression().fit(Xstd, yy).score(Xstd, yy)
    print(f"{lab}: OLS full-model R^2 = {ols_r2:.3f}")


def yfield(j):
    return r"$\mathrm{Re}\{\Psi\}$" if j < 25 else (r"$\mathrm{Im}\{\Psi\}$" if j < 50 else r"$U$")


def yalt(j):
    lev = j if j < 25 else (j - 25 if j < 50 else j - 50)
    return DZ * (lev + 1)


alphas = np.logspace(-2.2, 0.3, 32)
panels = []
for idx, label in [(1, "PC2"), (2, "PC3")]:
    y = pcs[:, idx]; ys = (y - y.mean()) / y.std()
    ytr, yte = ys[:ntr], ys[ntr:]
    C, nnz, r2 = [], [], []
    for a in alphas:
        las = Lasso(alpha=a, max_iter=4000, tol=1e-3).fit(Xtr, ytr)
        C.append(las.coef_); nnz.append(int((las.coef_ != 0).sum()))
        r2.append(r2_score(yte, las.predict(Xte)))
    # full-model R^2 (least regularized fit)
    full_r2 = r2[0]
    print(f"{label}: full R^2={full_r2:.3f}")
    panels.append(dict(label=label, C=np.array(C).T, nnz=np.array(nnz), r2=np.array(r2)))

combined = panels[0]["nnz"] + panels[1]["nnz"]
last = int(np.max(np.where(combined > 0)[0]))
for p in panels:
    p["C"] = p["C"][:, :last + 1]; p["nnz"] = p["nnz"][:last + 1]; p["r2"] = p["r2"][:last + 1]
ncol = last + 1
gvmax = max(np.abs(p["C"]).max() for p in panels)

fig = plt.figure(figsize=(17, 11), layout="constrained")
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.04], height_ratios=[5, 1.3])
ax_h = [fig.add_subplot(gs[0, 0])]
ax_h.append(fig.add_subplot(gs[0, 1], sharey=ax_h[0]))
ax_r = [fig.add_subplot(gs[1, c], sharex=ax_h[c]) for c in range(2)]
cax = fig.add_subplot(gs[0, 2])
xt = list(range(0, ncol, 3))
yt = [0, 12, 25, 37, 50, 62, 74]
im = None
for c, p in enumerate(panels):
    h, r = ax_h[c], ax_r[c]
    im = h.imshow(p["C"], aspect="auto", cmap="RdBu_r", vmin=-gvmax, vmax=gvmax,
                  interpolation="nearest", origin="lower")
    h.set_title(f"{p['label']}: surviving LASSO terms vs regularization", fontsize=16)
    for start in (25, 50):
        h.axhline(start - 0.5, color="k", lw=1)
    h.set_yticks(yt)
    h.set_yticklabels([f"{yfield(j)} {yalt(j):.0f} km" for j in yt], fontsize=15)
    h.tick_params(axis="x", labelbottom=False)
    r.plot(range(ncol), p["r2"], "o-", color="black", ms=3, lw=1.5)
    r.set_ylim(0, 1.0); r.set_ylabel("$R^2$", fontsize=16)
    r.set_xlabel("surviving terms (regularization increases $\\rightarrow$)", fontsize=16)
    r.set_xticks(xt); r.set_xticklabels([str(int(p["nnz"][i])) for i in xt], fontsize=13)
    r.grid(alpha=0.3)
fig.colorbar(im, cax=cax, label="coefficient")
fig.suptitle("LASSO path: which physical variables survive as the model is forced sparser",
             fontsize=17)
fig.savefig(os.path.join(OUT_DIR, "lasso_path_heatmap.png"), dpi=180)
plt.close(fig)
print("Saved Fig 11 -> lasso_path_heatmap.png")
print("ALL DONE")



LASSO interpretation of PC2 / PC3


r(PC1, U30) = 0.976
r(PC3, IHF) = 0.653   partial (control U30) = 0.408


PC2: OLS full-model R^2 = 0.916
PC3: OLS full-model R^2 = 0.946


PC2: full R^2=0.874


PC3: full R^2=0.926


Saved Fig 11 -> lasso_path_heatmap.png
ALL DONE
